In [1]:
%run 00_functions.ipynb

c:\Users\nyy32\OneDrive\Documents\UM_Master of Artifcial Inteligence\Thesis\Github\Cryptocurrency Forecasting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 07 — LLM Insights

Generate a textual interpretation of the latest saved LSTM forecast using the saved Top 10 news from `06_news_retrieval`.

**Model:** `gemini-3.5-flash` (single model; no automatic fallback).

**Important**
- This notebook does not rerun news retrieval or change the LSTM forecast.
- The saved news article content is used as input. URL Context was tested, but some Yahoo Finance URLs were blocked from automated retrieval, so URLs alone are not a reliable substitute for the saved article content.
- Run the two API-call cells deliberately. Each successful call may consume API quota.
- The two batches are used to keep each request smaller. Their news and risk sections are combined in a review draft; the draft is not automatically treated as the final approved report.

## 1. Imports, environment, and paths

In [2]:
# Load the project API key without printing its value.
env_path = Path("../data/env/.env")
load_dotenv(env_path)

api_key = os.getenv("GEMINI_API_KEY")
assert api_key, f"GEMINI_API_KEY was not found. Check: {env_path.resolve()}"

client = genai.Client(api_key=api_key)

MODEL_ID = "gemini-3.5-flash"
output_dir = Path("../data/llm")
output_dir.mkdir(parents=True, exist_ok=True)

print("Gemini client configured.")
print("Model:", MODEL_ID)
print("API key loaded:", bool(api_key))

Gemini client configured.
Model: gemini-3.5-flash
API key loaded: True


## 2. Load the saved forecast and Top 10 news

In [ ]:
prediction_path = Path("../data/lstm/processed/bitcoin_daily_predictions.csv")
news_path = Path("../data/news/processed/bitcoin_news_retrieved_top10.csv")

predictions_df = pd.read_csv(prediction_path)
news_df = pd.read_csv(news_path)

required_prediction_columns = {
    "forecast_date", "prediction_created_at", "predicted_close"
}
required_news_columns = {
    "similarity_rank", "title", "source", "published_at_utc",
    "link", "content", "content_status"
}

assert required_prediction_columns.issubset(predictions_df.columns), (
    f"Missing forecast columns: {required_prediction_columns - set(predictions_df.columns)}"
)
assert required_news_columns.issubset(news_df.columns), (
    f"Missing news columns: {required_news_columns - set(news_df.columns)}"
)

news_df = news_df.sort_values("similarity_rank").copy()
assert len(news_df) == 10, f"Expected 10 saved news articles; found {len(news_df)}"
assert news_df["link"].notna().all(), "At least one news URL is missing."

latest_prediction = predictions_df.sort_values("forecast_date").iloc[-1]

forecast_context = f"""
Forecast date: {latest_prediction['forecast_date']}
Prediction created at: {latest_prediction['prediction_created_at']}
Predicted Bitcoin closing price: {float(latest_prediction['predicted_close']):.2f} USD
Actual closing price: Not available yet
Prediction error: Not available yet
""".strip()

display(predictions_df.sort_values("forecast_date").tail(1))
display(news_df[["similarity_rank", "title", "source", "published_at_utc", "link", "content_status"]])

print("Forecast date:", latest_prediction["forecast_date"])
print("Predicted close:", f"{float(latest_prediction['predicted_close']):,.2f} USD")
print("News articles:", len(news_df))
print("News with saved content:", int(news_df["content"].fillna("").astype(str).str.strip().ne("").sum()))

,forecast_date,prediction_created_at,predicted_close,actual_close,prediction_error,absolute_error
1,2026-10-12,2026-10-12 13:49:25,81637.734375,NaN,NaN,NaN


,similarity_rank,title,source,published_at_utc,link,content_status
0,1,Bitcoin Could Be Entering a New Bull Market De...,BeInCrypto,2026-10-11 19:22:47+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
1,2,AI Price Prediction: Insights from Grok on Bit...,24/7 Wall St.,2026-10-11 12:12:14+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
2,3,"Bitcoin and ethereum prices today, Friday, Oct...",Yahoo Personal Finance,2026-10-09 11:33:50+00:00,https://finance.yahoo.com/personal-finance/inv...,summary_and_content
3,4,"Bitcoin rebounds above $83,000 after weekly sl...",Investing.com,2026-10-10 04:29:57+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
4,5,"Bitcoin Rebounds After Rough Week, But Traders...",decrypt,2026-10-09 17:36:26+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
5,6,"Bitcoin struggles above $83,000 a year after $...",Investing.com,2026-10-11 05:54:26+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
6,7,"Bitcoin, Strategy, and Crypto Stocks Rebound a...",Barrons.com,2026-10-09 12:21:00+00:00,https://finance.yahoo.com/m/fdafdedd-9b80-3ca6...,content_only
7,8,Bitcoin Tests Average Buyer’s Break-Even As ET...,Stocktwits,2026-10-10 17:34:25+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
8,9,Weekly Wrap: Rising Treasury Yields Sink Bitcoin,CryptoProwl,2026-10-09 20:52:00+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content
9,10,"Bitcoin holds ground above $82k, heads for wee...",Investing.com,2026-10-09 07:05:55+00:00,https://finance.yahoo.com/markets/crypto/artic...,summary_and_content


Forecast date: 2026-10-12
Predicted close: 81,637.73 USD
News articles: 10
News with saved content: 10


## 3. Define the output requirements

In [4]:
system_prompt = """
You are a financial news interpretation assistant for a Bitcoin forecasting system.

Interpret the supplied LSTM forecast using the saved retrieved news. Do not generate,
modify, or recalculate the numerical forecast.

Rules:
1. Treat the LSTM forecast as the primary quantitative result.
2. Use news as market context; do not claim that news directly caused the LSTM prediction.
3. Use only facts supported by the supplied forecast and saved news content.
4. Do not invent events, prices, statistics, sources, or explanations.
5. Distinguish reported facts from interpretations and potential risks.
6. State uncertainty when evidence is insufficient or conflicting.
7. Do not invent a numerical confidence score or calibrated probability.
8. Preserve source names and URLs when referring to news.
9. Do not present the output as guaranteed returns or financial advice.
10. If an article's saved content is short or incomplete, do not infer missing details from its title or URL.

Generate exactly these four sections:
1. Forecasting Result — forecast date and predicted closing price; make clear it is a prediction, not actual price.
2. Confidence — explain uncertainty in words, without a numerical score.
3. Key Market News — summarize relevant retrieved news and cite source names and URLs.
4. Potential Risks — identify supported risks, separate facts from possible implications, and state evidence limitations.

Write clearly and concisely in Markdown.
""".strip()

print("System prompt ready:", bool(system_prompt))
print("System prompt characters:", len(system_prompt))

System prompt ready: True
System prompt characters: 1458


## 4. Prepare two smaller requests

In [5]:
def make_news_context(batch_df):
    records = []
    for _, row in batch_df.iterrows():
        content = str(row["content"]) if pd.notna(row["content"]) else ""
        records.append(
            f"News {int(row['similarity_rank'])}\n"
            f"Title: {row['title']}\n"
            f"Source: {row['source']}\n"
            f"Published at (UTC): {row['published_at_utc']}\n"
            f"Content status: {row['content_status']}\n"
            f"Saved article content:\n{content}\n"
            f"Source URL: {row['link']}"
        )
    return "\n\n".join(records)

batch1_df = news_df.iloc[:5].copy()
batch2_df = news_df.iloc[5:].copy()

def make_user_prompt(batch_df, batch_label):
    return f"""
Interpret the following Bitcoin forecast using only the supplied forecast and saved news.

LSTM forecast:
{forecast_context}

Retrieved news ({batch_label}):
{make_news_context(batch_df)}

Do not invent facts. Do not claim that news directly caused the LSTM forecast.
Include the source name and URL for each news item discussed.
Return all four required sections.
""".strip()

batch1_user_prompt = make_user_prompt(batch1_df, "Batch 1, similarity ranks 1–5")
batch2_user_prompt = make_user_prompt(batch2_df, "Batch 2, similarity ranks 6–10")

for label, batch_df, prompt in [
    ("Batch 1", batch1_df, batch1_user_prompt),
    ("Batch 2", batch2_df, batch2_user_prompt),
]:
    print(f"{label}: ranks {batch_df['similarity_rank'].tolist()}")
    print(f"  Articles: {len(batch_df)}")
    print(f"  Prompt characters: {len(prompt):,}")
print("API called: No")

Batch 1: ranks [1, 2, 3, 4, 5]
  Articles: 5
  Prompt characters: 22,221
Batch 2: ranks [6, 7, 8, 9, 10]
  Articles: 5
  Prompt characters: 18,076
API called: No


## 5. Generate Batch 1 — run deliberately

In [6]:
# This cell makes one API request.
response_batch1 = client.models.generate_content(
    model=MODEL_ID,
    contents=batch1_user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        max_output_tokens=2048,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

batch1_insight = response_batch1.text
if not batch1_insight:
    raise RuntimeError(
        f"Batch 1 returned no text. Finish reason / candidates: "
        f"{response_batch1.candidates}"
    )

print("Model:", MODEL_ID)
print("Batch 1 generated.")
print("Finish reason:", response_batch1.candidates[0].finish_reason if response_batch1.candidates else "Unknown")
print(batch1_insight)

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 200 OK"


Model: gemini-3.5-flash
Batch 1 generated.
Finish reason: FinishReason.STOP
### 1. Forecasting Result
* **Forecast Date:** October 12, 2026
* **Prediction Timestamp:** 2026-10-12 13:49:25 UTC
* **Predicted Bitcoin Closing Price:** $81,637.73 USD

*Disclaimer: This value represents a model-based quantitative prediction generated by an LSTM network, not an actual market price or guaranteed financial return. The actual closing price and prediction error are currently not available.*

---

### 2. Confidence
The model relies strictly on quantitative primary data, leaving its statistical performance subject to market volatility. While technical patterns—such as Bitcoin holding above its 50-week moving average ($78,800)—suggest longer-term strength, short-term indicators reflect substantial uncertainty. 

The market has experienced wide price swings, recently moving between an October high near $87,249 and lows dipping toward $80,000. Additionally, options and derivative markets reflect ongoi

## 6. Generate Batch 2 — run deliberately

In [12]:
# This cell makes one API request.
response_batch2 = client.models.generate_content(
    model=MODEL_ID,
    contents=batch2_user_prompt,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        max_output_tokens=2048,
        thinking_config=types.ThinkingConfig(thinking_budget=0),
    ),
)

batch2_insight = response_batch2.text
if not batch2_insight:
    raise RuntimeError(
        f"Batch 2 returned no text. Finish reason / candidates: "
        f"{response_batch2.candidates}"
    )

print("Model:", MODEL_ID)
print("Batch 2 generated.")
print("Finish reason:", response_batch2.candidates[0].finish_reason if response_batch2.candidates else "Unknown")
print(batch2_insight)

INFO: AFC is enabled with max remote calls: 10.
INFO: HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:generateContent "HTTP/1.1 429 Too Many Requests"


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 7h58m56.335395418s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.6-fla

## 7. Save the two original batch responses

In [ ]:
# Run after both API-call cells have succeeded.
assert "batch1_insight" in globals(), "Run the Batch 1 generation cell first."
assert "batch2_insight" in globals(), "Run the Batch 2 generation cell first."

batch1_path = output_dir / "bitcoin_llm_insights_batch1.txt"
batch2_path = output_dir / "bitcoin_llm_insights_batch2.txt"

batch1_path.write_text(batch1_insight.strip() + "\n", encoding="utf-8")
batch2_path.write_text(batch2_insight.strip() + "\n", encoding="utf-8")

print("Batch 1 saved:", batch1_path.resolve())
print("Batch 2 saved:", batch2_path.resolve())
print("Batch 1 characters:", len(batch1_insight))
print("Batch 2 characters:", len(batch2_insight))

## 8. Validate and extract the four sections

In [ ]:
section_names = [
    "Forecasting Result",
    "Confidence",
    "Key Market News",
    "Potential Risks",
]

section_pattern = re.compile(
    r"(?m)^#{1,3}\s*\d*\.?\s*"
    r"(Forecasting Result|Confidence|Key Market News|Potential Risks)\s*:?.*$"
)

def extract_sections(text):
    matches = list(section_pattern.finditer(text))
    sections = {}
    for i, match in enumerate(matches):
        name_match = re.search(
            r"(Forecasting Result|Confidence|Key Market News|Potential Risks)",
            match.group(0),
        )
        name = name_match.group(1)
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        sections[name] = text[start:end].strip(" \n-")
    return sections

sections1 = extract_sections(batch1_insight)
sections2 = extract_sections(batch2_insight)

for label, sections in [("Batch 1", sections1), ("Batch 2", sections2)]:
    missing = [name for name in section_names if not sections.get(name)]
    print(f"{label} sections:", list(sections.keys()))
    print(f"{label} missing sections:", missing)
    if missing:
        print(f"WARNING: Review {label}; required sections are missing or empty.")

print("API called: No")

## 9. Build a unified review draft

In [ ]:
# Keep one shared forecast/confidence section, and retain the news and risk details
# from both batches under the corresponding main sections.
for label, sections in [("Batch 1", sections1), ("Batch 2", sections2)]:
    missing = [name for name in section_names if not sections.get(name)]
    assert not missing, f"{label} is missing required sections: {missing}"

sources_section = "\n\n".join(
    f"- **News {int(row['similarity_rank'])}: {row['title']}**\n"
    f"  - Source: {row['source']}\n"
    f"  - Published at (UTC): {row['published_at_utc']}\n"
    f"  - URL: {row['link']}\n"
    f"  - Saved content status: {row['content_status']}"
    for _, row in news_df.iterrows()
)

unified_report = f"""# Bitcoin LLM Insights

## 1. Forecasting Result

{sections1['Forecasting Result']}

## 2. Confidence

{sections1['Confidence']}

## 3. Key Market News

### Batch 1 — similarity ranks 1–5

{sections1['Key Market News']}

### Batch 2 — similarity ranks 6–10

{sections2['Key Market News']}

## 4. Potential Risks

### Batch 1 — similarity ranks 1–5

{sections1['Potential Risks']}

### Batch 2 — similarity ranks 6–10

{sections2['Potential Risks']}

## Sources

{sources_section}
"""

review_path = output_dir / "bitcoin_llm_insights_unified_review.txt"
review_path.write_text(unified_report.strip() + "\n", encoding="utf-8")

print("Unified review saved:", review_path.resolve())
print("Characters:", len(unified_report))
print("Sources included:", len(news_df))
print("API called: No")

## 10. Check report structure and source URLs

In [ ]:
saved_review = review_path.read_text(encoding="utf-8")

required_headings = [
    "## 1. Forecasting Result",
    "## 2. Confidence",
    "## 3. Key Market News",
    "## 4. Potential Risks",
    "## Sources",
]
positions = [saved_review.find(heading) for heading in required_headings]

print("Heading checks:")
for heading, position in zip(required_headings, positions):
    print(f"{heading}: {'FOUND' if position >= 0 else 'MISSING'}")

print(
    "Correct main-section order:",
    all(position >= 0 for position in positions)
    and positions == sorted(positions),
)

missing_urls = [
    str(row["link"])
    for _, row in news_df.iterrows()
    if str(row["link"]) not in saved_review
]

print("News URLs missing:", len(missing_urls))
print("News URLs present:", len(news_df) - len(missing_urls), "/", len(news_df))
if missing_urls:
    print("Missing URLs:")
    for url in missing_urls:
        print("-", url)

print("API called: No")